In [1]:
pip install timm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import KFold
import pandas as pd
import numpy as np
import cv2
from tqdm import tqdm
import math
import albumentations as A
from albumentations.pytorch import ToTensorV2
from transformers import AutoModel, AutoConfig

# ==================== CONFIGURATION ====================

class Config:
    # Paths
    train_csv = '/kaggle/input/csiro-biomass/train.csv'
    train_dir = '/kaggle/input/csiro-biomass/train'
    save_dir = 'dinov2_checkpoints'
    
    # Model
    backbone = 'timm/vit_large_patch16_dinov3.lvd1689m'  
    img_size = 448
    num_ssm_layers = 4
    dropout = 0.1
    
    # Training
    n_folds = 5
    batch_size = 1  # Safe for T4/P100
    stage1_epochs = 8   # Frozen
    stage2_epochs = 15  # Unfrozen
    stage1_lr = 1e-3
    stage2_lr = 5e-6
    
    # Other
    num_workers = 2
    seed = 42
    device = 'cuda' if torch.cuda.is_available() else 'cpu'

CFG = Config()

# Set seed
torch.manual_seed(CFG.seed)
np.random.seed(CFG.seed)

print(f"{'='*70}")
print(f"BIOMASS PREDICTION WITH DINOv2")
print(f"{'='*70}")
print(f"Device: {CFG.device}")
print(f"Batch size: {CFG.batch_size}")
print(f"Image size: {CFG.img_size}")
print(f"{'='*70}\n")

# ==================== WEIGHTED R² METRIC ====================

def calculate_weighted_r2(y_true, y_pred):
    """Competition metric: Weighted R²"""
    if torch.is_tensor(y_true):
        y_true = y_true.cpu().numpy()
    if torch.is_tensor(y_pred):
        y_pred = y_pred.cpu().numpy()
    
    weights = np.array([0.1, 0.1, 0.1, 0.2, 0.5])  # Per competition rules
    
    y_true_flat = y_true.flatten()
    y_pred_flat = y_pred.flatten()
    
    N = y_true.shape[0]
    weights_flat = np.tile(weights, N)
    
    y_w_mean = np.sum(weights_flat * y_true_flat) / np.sum(weights_flat)
    ss_res = np.sum(weights_flat * (y_true_flat - y_pred_flat) ** 2)
    ss_tot = np.sum(weights_flat * (y_true_flat - y_w_mean) ** 2)
    
    r2 = 1 - (ss_res / (ss_tot + 1e-8))
    return r2

# ==================== SSM COMPONENTS ====================

class SimpleSSM(nn.Module):
    """Simplified State Space Model"""
    def __init__(self, d_model, d_state=16):
        super().__init__()
        self.in_proj = nn.Linear(d_model, d_model * 2)
        self.conv1d = nn.Conv1d(d_model, d_model, kernel_size=4, padding=3, groups=d_model)
        self.dt_proj = nn.Linear(d_model, d_model)
        self.gate = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)
    
    def forward(self, x):
        x, gate = self.in_proj(x).chunk(2, dim=-1)
        x = x.transpose(1, 2)
        x = self.conv1d(x)[..., :x.shape[-1]].transpose(1, 2)
        x = F.silu(x)
        delta = torch.sigmoid(self.dt_proj(x))
        x = x * delta * torch.sigmoid(self.gate(gate))
        return self.out_proj(x)

class SSMBlock(nn.Module):
    """SSM block with residual connection"""
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_model)
        self.ssm = SimpleSSM(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        x = x + self.ssm(self.norm1(x))
        x = x + self.ffn(self.norm2(x))
        return x

# ==================== MAIN MODEL ====================

class DINOv3BiomassModel(nn.Module):
    def __init__(self):
        super().__init__()
        
        print(f"Loading DINOv3: {CFG.backbone}...")
        
        # 1. Load DINOv3 from Hugging Face
        self.backbone = AutoModel.from_pretrained(CFG.backbone)
        #self.backbone.set_grad_checkpointing(True) # <--- CRITICAL for saving memory
        
        # 2. Feature Dimension
        self.feature_dim = 1024 
        print(f"✅ DINOv3 Loaded! Dim: {self.feature_dim}")

        # --- MISSING PART RESTORED (Positional Encoding) ---
        max_len = 5000
        pe = torch.zeros(max_len, self.feature_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, self.feature_dim, 2).float() * (-math.log(10000.0) / self.feature_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        # ---------------------------------------------------

        # 4. SSM Layers
        self.ssm_layers = nn.ModuleList([
            SSMBlock(self.feature_dim, CFG.dropout) 
            for _ in range(CFG.num_ssm_layers)
        ])
        self.norm = nn.LayerNorm(self.feature_dim)
        
        # 5. Heads
        hidden = 512
        def make_head():
            return nn.Sequential(
                nn.Linear(self.feature_dim, hidden),
                nn.ReLU(),
                nn.Dropout(CFG.dropout),
                nn.Linear(hidden, 1),
                nn.Softplus()
            )
        
        self.head_green = make_head()
        self.head_dead = make_head()
        self.head_clover = make_head()
        
    def forward(self, x):
        # 1. Resize
        if x.shape[2:] != (CFG.img_size, CFG.img_size):
            x = F.interpolate(x, size=(CFG.img_size, CFG.img_size), mode='bilinear')
        
        # 2. Backbone
        outputs = self.backbone(pixel_values=x)
        features = outputs.last_hidden_state
        
        # 3. Remove 5 Tokens (CLS + Registers)
        features = features[:, 1:, :] 
        
        # 4. Add Positional Encoding
        if hasattr(self, 'pe'):
             features = features + self.pe[:, :features.size(1), :]
        
        # 5. SSM
        for layer in self.ssm_layers:
            features = layer(features)
            
        # 6. Global Pool & Predict
        features = self.norm(features)
        feat = features.mean(dim=1)
        
        green = self.head_green(feat)
        dead = self.head_dead(feat)
        clover = self.head_clover(feat)
        gdm = green + clover
        total = green + dead + clover
        
        return torch.cat([green, dead, clover, gdm, total], dim=1)
    
    # Helper for XGBoost (Passive)
    def extract_features(self, x):
        if x.shape[2:] != (CFG.img_size, CFG.img_size):
            x = F.interpolate(x, size=(CFG.img_size, CFG.img_size), mode='bilinear')
            
        outputs = self.backbone(pixel_values=x)
        features = outputs.last_hidden_state[:, 5:, :] 
        
        # Add PE here too if you want context-aware features for XGBoost
        if hasattr(self, 'pe'):
             features = features + self.pe[:, :features.size(1), :]

        for layer in self.ssm_layers:
            features = layer(features)
            
        features = self.norm(features)
        return features.mean(dim=1)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False
        print("🔒 Backbone frozen")
    
    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True
        print("🔓 Backbone unfrozen")
# ==================== DATASET ====================

class BiomassDataset(Dataset):
    def __init__(self, df, img_dir=CFG.train_dir, transform=None):
        self.image_paths = df['image_path'].unique()
        self.img_dir = img_dir
        self.transform = transform
        
        # Build target lookup
        self.targets = {}
        for img_path in self.image_paths:
            img_data = df[df['image_path'] == img_path]
            self.targets[img_path] = torch.tensor([
                img_data[img_data['target_name'] == 'Dry_Green_g']['target'].values[0],
                img_data[img_data['target_name'] == 'Dry_Dead_g']['target'].values[0],
                img_data[img_data['target_name'] == 'Dry_Clover_g']['target'].values[0],
                img_data[img_data['target_name'] == 'GDM_g']['target'].values[0],
                img_data[img_data['target_name'] == 'Dry_Total_g']['target'].values[0],
            ], dtype=torch.float32)
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        clean_path = img_path.replace('train/', '')
        full_path = os.path.join(self.img_dir, clean_path)
        
        img = cv2.imread(full_path)
        
        if img is None:
            img = np.zeros((1000, 2000, 3), dtype=np.uint8)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            img = self.transform(image=img)['image']
        
        return img, self.targets[img_path]

# ==================== TRANSFORMS ====================

def get_transforms(mode='train'):
    if mode == 'train':
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.Rotate(limit=10, p=0.3),
            A.RandomBrightnessContrast(p=0.4),
            A.HueSaturationValue(p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        return A.Compose([
            A.Resize(CFG.img_size, CFG.img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])

# ==================== TRAINING FUNCTIONS ====================

def train_epoch(model, loader, criterion, optimizer, scaler=None):
    model.train()
    losses = []
    
    pbar = tqdm(loader, desc='Training', leave=False)
    for images, targets in pbar:
        images = images.to(CFG.device)
        targets = targets.to(CFG.device)
        
        optimizer.zero_grad()
        
        if scaler:
            with torch.amp.autocast('cuda'):
                preds = model(images)
                loss = criterion(preds, targets)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            preds = model(images)
            loss = criterion(preds, targets)
            loss.backward()
            optimizer.step()
        
        losses.append(loss.item())
        pbar.set_postfix({'loss': f'{loss.item():.3f}'})
    
    return np.mean(losses)

def validate(model, loader, criterion):
    model.eval()
    losses = []
    all_preds, all_targets = [], []
    
    with torch.no_grad():
        for images, targets in tqdm(loader, desc='Validating', leave=False):
            images = images.to(CFG.device)
            targets = targets.to(CFG.device)
            preds = model(images)
            loss = criterion(preds, targets)
            
            losses.append(loss.item())
            all_preds.append(preds.cpu())
            all_targets.append(targets.cpu())
    
    all_preds = torch.cat(all_preds)
    all_targets = torch.cat(all_targets)
    
    r2 = calculate_weighted_r2(all_targets, all_preds)
    mae = torch.abs(all_preds - all_targets).mean(dim=0).numpy()
    
    return np.mean(losses), r2, mae

# ==================== MAIN TRAINING ====================

def main():
    os.makedirs(CFG.save_dir, exist_ok=True)
    
    # Load data
    print(f"Loading {CFG.train_csv}...")
    train_df = pd.read_csv(CFG.train_csv)
    unique_images = train_df['image_path'].unique()
    print(f"Found {len(unique_images)} images\n")
    
    # K-Fold CV
    kfold = KFold(n_splits=CFG.n_folds, shuffle=True, random_state=CFG.seed)
    cv_results = []
    
    for fold, (train_idx, val_idx) in enumerate(kfold.split(unique_images)):
        print(f'\n{"="*70}')
        print(f'FOLD {fold + 1}/{CFG.n_folds}')
        print(f'{"="*70}\n')
        
        # Split data
        train_imgs = unique_images[train_idx]
        val_imgs = unique_images[val_idx]
        
        train_fold_df = train_df[train_df['image_path'].isin(train_imgs)]
        val_fold_df = train_df[train_df['image_path'].isin(val_imgs)]
        
        # Datasets
        train_dataset = BiomassDataset(train_fold_df, transform=get_transforms('train'))
        val_dataset = BiomassDataset(val_fold_df, transform=get_transforms('val'))
        
        train_loader = DataLoader(
            train_dataset, 
            batch_size=CFG.batch_size,
            shuffle=True, 
            num_workers=CFG.num_workers,
            pin_memory=True
        )
        val_loader = DataLoader(
            val_dataset,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
            pin_memory=True
        )
        
        print(f"Train: {len(train_imgs)} images")
        print(f"Val: {len(val_imgs)} images\n")
        
        # Model
        model = DINOv3BiomassModel().to(CFG.device)
        if fold == 0:
            config_path = os.path.join(CFG.save_dir, 'dinov3_config')
            model.backbone.config.save_pretrained(config_path)
            print(f"✅ Saved DINOv3 Config to: {config_path}")
        criterion = nn.MSELoss()
        best_r2 = -float('inf')
        
        # ===== STAGE 1: FREEZE BACKBONE =====
        print(f"{'='*70}")
        print("STAGE 1: Frozen backbone (fast warm-up)")
        print(f"{'='*70}\n")
        
        model.freeze_backbone()
        
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=CFG.stage1_lr,
            weight_decay=1e-5
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, 
            T_max=CFG.stage1_epochs
        )
        
        for epoch in range(CFG.stage1_epochs):
            print(f"Epoch {epoch+1}/{CFG.stage1_epochs}")
            
            train_loss = train_epoch(model, train_loader, criterion, optimizer)
            val_loss, val_r2, mae = validate(model, val_loader, criterion)
            scheduler.step()
            
            print(f"  Loss: {train_loss:.3f} → {val_loss:.3f}")
            print(f"  R²: {val_r2:.4f}")
            print(f"  MAE Total: {mae[4]:.1f}g\n")
            
            if val_r2 > best_r2:
                best_r2 = val_r2
                torch.save(
                    model.state_dict(),
                    os.path.join(CFG.save_dir, f'stage1_fold{fold}.pth')
                )
                print(f"  ✅ Saved (R²={val_r2:.4f})\n")
        
        # ===== STAGE 2: FINE-TUNE ALL =====
        print(f"\n{'='*70}")
        print("STAGE 2: Fine-tune all (slow polish)")
        print(f"{'='*70}\n")
        
        model.unfreeze_backbone()
        
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=CFG.stage2_lr,
            weight_decay=1e-5
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=CFG.stage2_epochs
        )
        scaler = torch.amp.GradScaler('cuda')
        
        for epoch in range(CFG.stage2_epochs):
            print(f"Epoch {epoch+1}/{CFG.stage2_epochs}")
            
            train_loss = train_epoch(model, train_loader, criterion, optimizer, scaler)
            val_loss, val_r2, mae = validate(model, val_loader, criterion)
            scheduler.step()
            
            print(f"  Loss: {train_loss:.3f} → {val_loss:.3f}")
            print(f"  R²: {val_r2:.4f}")
            print(f"  MAE: G={mae[0]:.1f} D={mae[1]:.1f} C={mae[2]:.1f} Tot={mae[4]:.1f}\n")
            
            if val_r2 > best_r2:
                best_r2 = val_r2
                torch.save(
                    model.state_dict(),
                    os.path.join(CFG.save_dir, f'best_fold{fold}.pth')
                )
                print(f"  ✅ BEST! (R²={val_r2:.4f})\n")
        
        cv_results.append(best_r2)
        print(f"🎯 Fold {fold+1} Best R²: {best_r2:.4f}")
        
        # Cleanup
        del model, optimizer, scheduler
        torch.cuda.empty_cache()
    
    # Final summary
    print(f"\n{'='*70}")
    print("FINAL RESULTS")
    print(f"{'='*70}\n")
    print(f"R² scores: {[f'{r:.4f}' for r in cv_results]}")
    print(f"Mean: {np.mean(cv_results):.4f} ± {np.std(cv_results):.4f}")
    print(f"Best fold: {np.argmax(cv_results) + 1}")
    print(f"\nModels saved in: {CFG.save_dir}/")
    print(f"✅ Training complete!")

if __name__ == '__main__':
    main()

BIOMASS PREDICTION WITH DINOv2
Device: cuda
Batch size: 1
Image size: 448

Loading /kaggle/input/csiro-biomass/train.csv...
Found 357 images


FOLD 1/5

Train: 285 images
Val: 72 images

Loading DINOv3: timm/vit_large_patch16_dinov3.lvd1689m...


config.json:   0%|          | 0.00/671 [00:00<?, ?B/s]

/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` 

model.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]

✅ DINOv3 Loaded! Dim: 1024
✅ Saved DINOv3 Config to: dinov2_checkpoints/dinov3_config
STAGE 1: Frozen backbone (fast warm-up)

🔒 Backbone frozen
Epoch 1/8


  Loss: 576.701 → 525.857
  R²: -0.1450
  MAE Total: 23.7g

  ✅ Saved (R²=-0.1450)

Epoch 2/8


... (110 outputs left out)

  Loss: 120.255 → 187.286
  R²: 0.7245
  MAE: G=9.6 D=9.9 C=7.4 Tot=12.0

Epoch 13/15


  Loss: 119.176 → 185.964
  R²: 0.7259
  MAE: G=9.5 D=9.9 C=7.3 Tot=12.0

Epoch 14/15


  Loss: 114.283 → 185.873
  R²: 0.7253
  MAE: G=9.4 D=9.9 C=7.3 Tot=12.0

Epoch 15/15


  Loss: 118.977 → 184.931
  R²: 0.7285
  MAE: G=9.5 D=9.9 C=7.3 Tot=12.0

🎯 Fold 5 Best R²: 0.7290

FINAL RESULTS

R² scores: ['0.7564', '0.7443', '0.7706', '0.7774', '0.7290']
Mean: 0.7555 ± 0.0175
Best fold: 4

Models saved in: dinov2_checkpoints/
✅ Training complete!
